# 03 - Ekstraksi Fitur NLP: VADER, Loughran-McDonald, TF-IDF

Input: `data/split/daily_kurs_split.csv` dan `data/split/articles_split.csv` (hasil notebook 02). Kamus Loughran-McDonald harus diunduh sendiri (Notre Dame SRAF, "Master Dictionary" versi CSV) dan diletakkan di `data/raw/`.

Output (semua di `data/feature_extracted/`):
- `features_lexicon.csv`: fitur VADER dan LM per hari bursa.
- `tfidf_X.npz`, `tfidf_vocab.csv`, `tfidf_index.csv`: matriks TF-IDF per hari.
- `model_dataset.csv`: satu tabel siap modeling (kurs + lag + fitur lexicon + komponen SVD dari TF-IDF).

Lexicon (VADER, LM) tidak dilatih dari data, jadi aman dihitung untuk semua artikel. Hanya TF-IDF yang di-fit, dan itu dilakukan pada hari-hari train saja.

In [ ]:
!pip install vaderSentiment

In [ ]:
import re
import numpy as np
import pandas as pd
from collections import Counter
from scipy import sparse
from tqdm.auto import tqdm
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

from pathlib import Path

def find_repo_root():
    """Cari folder root repo (yang berisi data/ dan src/), dari mana pun notebook dijalankan."""
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "data").is_dir() and (d / "src").is_dir():
            return d
    raise FileNotFoundError(f"Root repo (berisi data/ dan src/) tidak ditemukan dari {here}")

ROOT = find_repo_root()
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
SPLIT_DIR = ROOT / "data" / "split"
FEATURE_DIR = ROOT / "data" / "feature_extracted"
for d in (PROCESSED_DIR, SPLIT_DIR, FEATURE_DIR):
    d.mkdir(parents=True, exist_ok=True)

LM_FILE = RAW_DIR / "Loughran-McDonald_MasterDictionary_1993-2024.csv"   # sesuaikan nama file unduhan
LM_CATEGORIES = ["Negative", "Positive", "Uncertainty", "Litigious", "Constraining"]

TFIDF_FIELD = "both"        # "title", "body", atau "both": bagian teks mana yang dipakai
TFIDF_MAX_FEATURES = 2000
TFIDF_MIN_DF = 5            # kata harus muncul di minimal segini hari (train)
TFIDF_MAX_DF = 0.8
SVD_COMPONENTS = 50

# Berita mana yang dipakai untuk memprediksi kurs_besok?
#  "sebelum_penetapan_besok"   : berita setelah penetapan JISDOR hari ini sampai sebelum penetapan besok
#  "sampai_penetapan_hari_ini" : hanya berita sampai penetapan hari ini (lebih konservatif)
NEWS_TIMING = "sebelum_penetapan_besok"

daily = pd.read_csv(SPLIT_DIR / "daily_kurs_split.csv", parse_dates=["date"])
arts = pd.read_csv(SPLIT_DIR / "articles_split.csv", parse_dates=["trading_date"])
arts["body_bow"] = arts["body_bow"].fillna("")
arts["title_bow"] = arts["title_bow"].fillna("")
print(daily.shape, arts.shape)

## 1. VADER

Judul dan isi diperlakukan berbeda. Judul pendek, jadi skor `compound` langsung dipakai. Isi artikel panjang, dan `compound` pada teks panjang cenderung menempel di -1 atau +1, sehingga tidak membedakan artikel. Karena itu isi dipecah per kalimat, dan yang diambil rata-rata skor kalimat serta porsi kalimat negatif (`compound` <= -0.05, ambang standar VADER).

Inputnya `title_clean` dan `body_clean`, karena VADER memakai huruf kapital, tanda baca, dan negasi.

In [ ]:
sia = SentimentIntensityAnalyzer()
SENT_SPLIT_RE = re.compile(r"(?<=[.!?])\s+|\n+")

def vader_title(text):
    return sia.polarity_scores(str(text))["compound"]

def vader_body(text):
    sents = [s for s in SENT_SPLIT_RE.split(str(text)) if len(s.split()) >= 3]
    if not sents:
        return 0.0, 0.0
    scores = np.array([sia.polarity_scores(s)["compound"] for s in sents])
    return scores.mean(), (scores <= -0.05).mean()

arts["vader_title"] = [vader_title(t) for t in tqdm(arts["title_clean"], desc="VADER judul")]
body_scores = [vader_body(t) for t in tqdm(arts["body_clean"], desc="VADER isi")]
arts["vader_body"] = [b[0] for b in body_scores]
arts["vader_neg_share"] = [b[1] for b in body_scores]

arts[["vader_title", "vader_body", "vader_neg_share"]].describe().round(3)

## 2. Loughran-McDonald

Kamus LM dibuat khusus untuk teks keuangan. Tiap kata punya kategori (negative, positive, uncertainty, litigious, constraining). Skor sebuah artikel = jumlah kata di kategori itu dibagi total kata artikel, sehingga artikel panjang tidak otomatis bernilai tinggi. `lm_net` = positive - negative.

Inputnya `body_bow` (huruf kecil, tanpa stopword, tanpa stemming), sesuai bentuk kata di kamus. Di file kamus, sebuah kata termasuk kategori kalau nilai kolomnya bukan 0 (isinya tahun kata itu ditambahkan).

In [ ]:
lm = pd.read_csv(LM_FILE)
lm["Word"] = lm["Word"].astype(str).str.lower()

present = [c for c in LM_CATEGORIES if c in lm.columns]
missing = [c for c in LM_CATEGORIES if c not in lm.columns]
if missing:
    print("Kolom tidak ada di file kamus, dilewati:", missing)

LM_SETS = {c.lower(): set(lm.loc[lm[c] > 0, "Word"]) for c in present}
print({k: len(v) for k, v in LM_SETS.items()})

def lm_scores(bow_text, n_words):
    counts = Counter(bow_text.split())
    denom = max(int(n_words), 1)
    out = {f"lm_{cat}": sum(v for w, v in counts.items() if w in words) / denom
           for cat, words in LM_SETS.items()}
    if "lm_positive" in out and "lm_negative" in out:
        out["lm_net"] = out["lm_positive"] - out["lm_negative"]
    return out

lm_df = pd.DataFrame([lm_scores(b, n) for b, n in tqdm(zip(arts["body_bow"], arts["n_words"]),
                                                       total=len(arts), desc="LM")])
arts = pd.concat([arts.reset_index(drop=True), lm_df], axis=1)
lm_df.describe().round(4)

## 3. Agregasi ke level hari bursa

Satu hari bisa punya banyak artikel, sedangkan kurs hanya satu angka. Ringkasan per hari:

- rata-rata semua skor,
- skor VADER isi paling negatif hari itu (`vader_body_min`), untuk menangkap satu berita bernada buruk yang tenggelam kalau hanya dirata-rata,
- jumlah artikel (`n_articles`).

Hari tanpa berita diisi 0 (netral) dan diberi penanda `no_news = 1` supaya model bisa membedakan "netral" dari "tidak ada berita". Cara pengisian ini keputusan tim, boleh diganti.

**Timing berita terhadap target.** Berita yang terbit setelah penetapan JISDOR hari ini dan sebelum penetapan besok adalah berita yang secara waktu bisa memengaruhi `kurs_besok`. Pengaturan `NEWS_TIMING`:

- `"sebelum_penetapan_besok"` (default): baris hari t memakai berita di jendela itu. Prediksi dianggap dibuat sesaat sebelum JISDOR besok ditetapkan, dengan kurs hari ini sudah diketahui. Ini harus ditulis jelas di laporan.
- `"sampai_penetapan_hari_ini"`: baris hari t hanya memakai berita sampai penetapan hari ini. Lebih konservatif, tapi berita jadi terlambat satu hari bursa dari kurs yang ingin dijelaskan, sehingga pengaruh berita bisa terlihat lebih kecil dari sebenarnya.

Membandingkan keduanya bisa menjadi satu eksperimen tersendiri di laporan.

Karena berita di baris t berasal dari hari berikutnya, baris terakhir tiap split dibuang (target dan beritanya sudah masuk split sesudahnya), supaya tidak ada informasi yang menyeberang batas train/val/test.

In [ ]:
lm_cols = list(lm_df.columns)

agg = arts.groupby("trading_date").agg(
    n_articles=("url", "size"),
    vader_title_mean=("vader_title", "mean"),
    vader_body_mean=("vader_body", "mean"),
    vader_body_min=("vader_body", "min"),
    vader_neg_share=("vader_neg_share", "mean"),
    **{f"{c}_mean": (c, "mean") for c in lm_cols},
)

days = daily.copy()
if NEWS_TIMING == "sebelum_penetapan_besok":
    days["news_date"] = days["date"].shift(-1)                 # jendela berita hari bursa berikutnya
    keep = days["split"] == days["split"].shift(-1)            # buang baris terakhir tiap split
    days = days[keep].reset_index(drop=True)
else:
    days["news_date"] = days["date"]
days = days.merge(agg, left_on="news_date", right_index=True, how="left")
feat_cols = list(agg.columns)
days["no_news"] = days["n_articles"].isna().astype(int)
days[feat_cols] = days[feat_cols].fillna(0)
days["n_articles"] = days["n_articles"].astype(int)

lex_cols = feat_cols + ["no_news"]
days[["date", "split"] + lex_cols].to_csv(FEATURE_DIR / "features_lexicon.csv", index=False)
print(days["no_news"].mean().round(3), "= porsi hari tanpa berita")
days[["date", "split"] + lex_cols].head()

## 4. TF-IDF

Satu hari = satu dokumen (semua artikel hari itu digabung). Vocabulary dan bobot IDF dipelajari **hanya dari hari-hari train**, lalu dipakai untuk mentransformasi val dan test. Bagian teks yang dipakai diatur lewat `TFIDF_FIELD` ("title", "body", "both"), jadi perbandingan judul vs isi penuh (pertanyaan di Tugas 2) tinggal mengganti satu nilai.

Matriks TF-IDF berdimensi ribuan, sedangkan jumlah hari bursa hanya sekitar 1.000-an, jadi mudah overfit. Karena itu ada versi ringkas lewat TruncatedSVD (fit juga hanya di train) yang mengubahnya jadi puluhan kolom.

In [ ]:
def make_text(row):
    if TFIDF_FIELD == "title":
        return row["title_bow"]
    if TFIDF_FIELD == "body":
        return row["body_bow"]
    return row["title_bow"] + " " + row["body_bow"]

arts["tfidf_text"] = arts.apply(make_text, axis=1)
docs = (arts.groupby("trading_date")["tfidf_text"].apply(" ".join)
            .reindex(days["news_date"]).fillna(""))
docs = docs.reset_index(drop=True)

train_mask = (days["split"] == "train").values

vec = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES, min_df=TFIDF_MIN_DF, max_df=TFIDF_MAX_DF,
    sublinear_tf=True, ngram_range=(1, 1),
)
vec.fit(docs[train_mask])
X = vec.transform(docs)             # semua hari, urutan sama dengan days
print("Matriks TF-IDF:", X.shape)

sparse.save_npz(FEATURE_DIR / "tfidf_X.npz", X)
pd.Series(vec.get_feature_names_out(), name="term").to_csv(FEATURE_DIR / "tfidf_vocab.csv", index=False)
days[["date", "split"]].to_csv(FEATURE_DIR / "tfidf_index.csv", index=False)

# istilah dengan bobot rata-rata tertinggi di train (cek kewajaran)
mean_w = np.asarray(X[train_mask].mean(axis=0)).ravel()
top = pd.Series(mean_w, index=vec.get_feature_names_out()).nlargest(20)
print("\n20 istilah dengan bobot rata-rata tertinggi (train):")
print(", ".join(top.index))

In [ ]:
n_comp = min(SVD_COMPONENTS, X.shape[1] - 1)
svd = TruncatedSVD(n_components=n_comp, random_state=42).fit(X[train_mask])
Z = svd.transform(X)
svd_cols = [f"tfidf_svd_{i+1}" for i in range(n_comp)]
svd_df = pd.DataFrame(Z, columns=svd_cols)

print(f"{n_comp} komponen SVD menjelaskan {svd.explained_variance_ratio_.sum():.1%} variansi (train)")

## 5. Gabungkan jadi dataset modeling

Satu baris per hari bursa. Belum ada scaling atau normalisasi di sini; lakukan di tahap modeling dengan `fit` hanya pada train.

In [ ]:
model_df = pd.concat([days.reset_index(drop=True), svd_df], axis=1)
model_df.to_csv(FEATURE_DIR / "model_dataset.csv", index=False)
print("Tersimpan di:", FEATURE_DIR)

print(model_df.shape)
print(model_df["split"].value_counts().to_dict())
model_df.head()

## 6. Cek cepat: korelasi fitur dengan return (train)

Korelasi Spearman antara tiap fitur lexicon dan `delta_besok` (kurs_besok - kurs_jisdor) di data train. Angkanya kemungkinan kecil (kurs harian memang sulit diprediksi). Ini hanya cek kewajaran arah, misalnya apakah sentimen negatif berhubungan dengan pelemahan rupiah, bukan bukti hipotesis.

In [ ]:
tr = model_df[model_df["split"] == "train"]
corr = tr[lex_cols + ["delta_besok"]].corr(method="spearman")["delta_besok"].drop("delta_besok")
print(corr.sort_values().round(3).to_string())